# What is the Pump It Up training label?

## Provenance, semantics and target structure

Short answer: status_group is best treated as a **survey-time administrative condition label**, not a timeless property, a failure outcome, or a direct repair-priority score. DrivenData documents three flat classes, but the source programme used operational state, repair need, hardware condition and breakdown duration together. The most defensible competition model therefore remains flat multiclass classification. Two binary trees and an ordinal encoding are useful hypotheses to test, not established ground truth.

This notebook separates documented fact, local empirical evidence, inference and unresolved unknowns. It uses only aggregate outputs; it does not reproduce competition rows.

## Course-aligned ten-step lifecycle

| Step | Treatment in this workstream |
|---:|---|
| 1. Define the goal and scope | Determine what the observed label meant, how it was produced, and which target structures are defensible. Success means an evidence-graded conclusion, not a leaderboard gain. |
| 2. Gather the data | Use the immutable competition files plus public primary and peer-reviewed provenance sources. No external data enters a competition model. |
| 3. Explore the data | Audit class counts, dates, collector identity, contemporaneous survey fields and indistinguishable duplicate records. |
| 4. Clean and preprocess the data | Use the existing validated modelling handoff and fold-fitted preprocessing. No target relabelling is performed. |
| 5. Select and engineer features | Reuse the frozen 36-predictor policy and existing initial feature engineering for diagnostic models. No new production feature is selected here. |
| 6. Define the machine-learning task | Compare flat nominal multiclass, a tentative ordinal view, two binary trees and a multiple-latent-axis interpretation. |
| 7. Partition the data | Reuse the frozen 80% development partition and its five predefined folds; leave the local test untouched. |
| 8. Select and train candidate methods | Fit the existing constrained decision-tree pipeline as a deliberately modest separability probe, not as a final model. |
| 9. Evaluate and interpret the results | Compare fold-level binary discrimination, flat accuracy, per-class recall and macro recall. |
| 10. Deploy and iterate | Deployment is not applicable: this is a diagnostic workstream. Record implications for later modelling, monitoring and label collection. |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    roc_auc_score,
)

stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'data' / 'TrainingSetValues.csv').is_file()
)
source_directory = str((stage_directory / 'src').resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from data_partitioning import make_cross_validation, partition_modelling_data
from model_evaluation import make_initial_decision_tree_pipeline
from modelling_data import prepare_modelling_data
from ordinal_target import (
    cumulative_class_probabilities,
    cumulative_threshold_predictions,
    ordinal_error_metrics,
    project_cumulative_probabilities,
    tune_cumulative_cutoffs,
)

data_directory = stage_directory / 'data'
raw_training = pd.read_csv(data_directory / 'TrainingSetValues.csv')
training_labels = pd.read_csv(data_directory / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(data_directory / 'TestSetValues.csv')

labelled = raw_training.merge(
    training_labels,
    on='id',
    how='inner',
    validate='one_to_one',
)
modelling_data = prepare_modelling_data(
    raw_training,
    training_labels,
    raw_competition,
)
partitioned = partition_modelling_data(modelling_data)
cross_validation = make_cross_validation(partitioned)
fold_splits = list(cross_validation.split())

CLASS_LABELS = [
    'functional',
    'functional needs repair',
    'non functional',
]
print(
    f'Loaded {len(labelled):,} labelled rows and '
    f'{len(raw_competition):,} competition rows; '
    f'frozen CV fingerprint {partitioned.cross_validation_fingerprint[:12]}…'
)

Loaded 59,400 labelled rows and 14,850 competition rows; frozen CV fingerprint bd7da743e9b4…


## 1–2. Goal, source evidence and provenance chain

### Documented fact

- [DrivenData's Taarifa page](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/page/24/) says the competition data came from the Taarifa waterpoints dashboard, which aggregated Tanzanian Ministry of Water data. This does **not** mean the labels were originally crowdsourced through Taarifa.
- The [World Bank's 2016 completion report](https://documents1.worldbank.org/curated/en/919091467777086225/pdf/ICR3737-P087154-Box396252B-PUBLIC-disclosed-7-1-16.pdf) says the Ministry engaged a local firm in 2011, which geotagged and collected basic data from about 75,000 water points during 2012–2013.
- The [Tanzania Ministry of Water's 2013 sector report](https://www.maji.go.tz/uploads/publications/en1568462815-2013_Water_Sector_Status_Report.pdf) records 75,777 points mapped by June 2013 and describes the mapping system as a planning and monitoring asset.
- A later [World Bank completion report](https://documents1.worldbank.org/curated/en/099042423190529784/pdf/P16175705053eb050ae2907c76d3ea463f.pdf) describes the 2013 national exercise as 74,250 points: 55% functional, 7% needing repair and 38% non-functional.
- Verplanke and Georgiadou's peer-reviewed [Wicked Water Points](https://doi.org/10.3390/ijgi6080244) traces the source WPMS. It reports that a water-point collector recorded STATUS in seven classes; the consultant separately produced a binary STATUS2 aggregation. It also reports a February 2014 release totalling 74,250 points after accounting for an omitted region, plus duplicate, identifier, definition and observer-quality problems.
- Their companion paper, [Tensions in Rural Water Governance](https://doi.org/10.3390/ijgi6090266), reports that the 2010–2013 baseline used a conditional status programme and local informants. Under that programme, hardware problems and some short-duration non-functionality could enter the repair-needed category; informal practice sometimes labelled short interruptions as functional.
- [DrivenData's problem page](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/page/25/) presents a simpler competition contract: functional means operational with no repair needed; functional needs repair means operational but needing repair; non functional means not operational.

### Strong inference, not a documented crosswalk

The exact 74,250-row total, the same 55/7/38 split, the feature correspondence, the collection dates and the constant collector name strongly indicate that DrivenData split the cleaned February 2014 WPMS release into 59,400 labelled and 14,850 unlabelled rows. Neither the competition page nor the cited programme documents provide a row-level lineage file or the exact transformation from source STATUS values to status_group, so this must remain an inference.

In [2]:
all_competition_ids = pd.concat(
    [raw_training['id'], raw_competition['id']],
    ignore_index=True,
)
target_counts = (
    training_labels['status_group']
    .value_counts()
    .reindex(CLASS_LABELS)
)

lineage_fingerprint = pd.DataFrame(
    {
        'observed': [
            len(raw_training),
            len(raw_competition),
            len(all_competition_ids),
            all_competition_ids.nunique(),
            bool(set(all_competition_ids) == set(range(74_250))),
            raw_training['recorded_by'].nunique(dropna=False),
            raw_training['recorded_by'].iloc[0],
        ],
        'interpretation': [
            'Labelled competition partition',
            'Unlabelled competition partition',
            'Exact match to the reported 2013/February-2014 WPM total',
            'No ID overlap or duplication across the two partitions',
            'Competition IDs are a complete new integer key, not source WPTCODEs',
            'One recorded_by value in labelled predictors',
            'Collector identity agrees with a consultant-produced extract',
        ],
    },
    index=[
        'training rows',
        'competition rows',
        'combined rows',
        'combined unique IDs',
        'IDs exactly 0..74249',
        'recorded_by distinct values',
        'recorded_by value',
    ],
)
display(lineage_fingerprint)

target_distribution = target_counts.rename('rows').to_frame()
target_distribution['share'] = target_distribution['rows'] / len(training_labels)
display(target_distribution.style.format({'share': '{:.2%}'}))

,observed,interpretation
training rows,59400,Labelled competition partition
competition rows,14850,Unlabelled competition partition
combined rows,74250,Exact match to the reported 2013/February-2014...
combined unique IDs,74250,No ID overlap or duplication across the two pa...
IDs exactly 0..74249,True,Competition IDs are a complete new integer key...
recorded_by distinct values,1,One recorded_by value in labelled predictors
recorded_by value,GeoData Consultants Ltd,Collector identity agrees with a consultant-pr...


,rows,share
status_group,,
functional,32259,54.31%
functional needs repair,4317,7.27%
non functional,22824,38.42%


## 3. What did the label mean at collection time?

The source evidence makes this a **reference-epoch observation**. The collector visited or located a water point, consulted local sources and recorded its then-current administrative functionality class. It was intended to support coverage estimates, maintenance planning and later monitoring. It was not a measured time-to-failure outcome and does not certify what happened after the visit.

The original meaning was less clean than the competition gloss:

| Aspect | Evidence-led reading |
|---|---|
| Unit | A mapped rural public water point, represented once in this extract unless source/data-processing duplication occurred. |
| Reference time | `date_recorded` is the documented row-entry date and best available reference-date proxy; the observation-to-entry lag is unknown. Dates are overwhelmingly 2011–2013. |
| Reporter | STATUS was attributed to a water-point collector in the source audit, drawing on observation and local informants. |
| Construct | Administrative functionality, combining whether service was available with repair condition and, in the source programme, breakdown duration. |
| Intended use | National coverage reporting, planning, budgeting, rehabilitation and later monitoring. |
| Not established | A laboratory test, independent engineering inspection, future failure label, repair cost, repair feasibility, or durable state. |

The seven source STATUS values included functional, functional needing repair, four overlapping duration-banded non-functional variants and an unqualified non-functional value. The exact competition reduction to three values is undocumented. The three published competition labels resemble a retained/cleaned STATUS view much more than the binary STATUS2 view, but that is still inference.

In [3]:
recorded_dates = pd.to_datetime(labelled['date_recorded'], errors='raise')
year_counts = recorded_dates.dt.year.value_counts().sort_index().rename('rows').to_frame()
year_counts['share'] = year_counts['rows'] / len(labelled)

date_profile = pd.DataFrame(
    {
        'value': [
            recorded_dates.min().date().isoformat(),
            recorded_dates.max().date().isoformat(),
            int(recorded_dates.dt.year.isin([2011, 2012, 2013]).sum()),
            int((~recorded_dates.dt.year.isin([2011, 2012, 2013])).sum()),
        ]
    },
    index=[
        'earliest recorded date',
        'latest recorded date',
        'rows recorded in 2011–2013',
        'earlier-date anomalies',
    ],
)
display(date_profile)
display(year_counts.style.format({'share': '{:.2%}'}))

,value
earliest recorded date,2002-10-14
latest recorded date,2013-12-03
rows recorded in 2011–2013,59369
earlier-date anomalies,31


,rows,share
date_recorded,,
2002,1,0.00%
2004,30,0.05%
2011,28674,48.27%
2012,6424,10.81%
2013,24271,40.86%


The 31 rows dated 2002 or 2004 are anomalies relative to the national 2011–2013 campaign. They may be carried-forward dates or data errors; this dataset cannot decide which. More importantly, every label ages from its own observation date. A model trained on this snapshot predicts the historic recorded class under the old reporting programme. Calling its output a current pump-status prediction requires a fresh observation-time feature set and a contemporary label definition.

## 4–5. Empirical label relationships, proxy risks and indistinguishable rows

Several predictors were collected in the same field exercise as STATUS. They can be legitimate contemporaneous evidence for reproducing the survey label, but some become unavailable or tautological in a genuine pre-visit prediction. Quantity and quality were also reported as judgement-heavy fields in the source audit. Their association with the target is therefore evidence about the **recording process** as well as about physical pumps.

In [4]:
def bias_corrected_cramers_v(left: pd.Series, right: pd.Series) -> float:
    table = pd.crosstab(left.astype('string').fillna('__missing__'), right)
    chi_squared = chi2_contingency(table, correction=False)[0]
    rows = table.to_numpy().sum()
    row_levels, column_levels = table.shape
    phi_squared = chi_squared / rows
    corrected_phi = max(
        0.0,
        phi_squared
        - ((column_levels - 1) * (row_levels - 1)) / (rows - 1),
    )
    corrected_rows = row_levels - ((row_levels - 1) ** 2) / (rows - 1)
    corrected_columns = (
        column_levels - ((column_levels - 1) ** 2) / (rows - 1)
    )
    return float(
        np.sqrt(corrected_phi / min(corrected_rows - 1, corrected_columns - 1))
    )

relationship_features = [
    'quantity',
    'water_quality',
    'extraction_type_class',
    'waterpoint_type_group',
    'region',
    'payment_type',
    'source_class',
    'management_group',
]
associations = pd.Series(
    {
        feature: bias_corrected_cramers_v(
            labelled[feature],
            labelled['status_group'],
        )
        for feature in relationship_features
    },
    name='bias-corrected Cramér V',
).sort_values(ascending=False)
display(associations.to_frame().style.format('{:.3f}'))

quantity_counts = pd.crosstab(
    labelled['quantity'],
    labelled['status_group'],
).reindex(columns=CLASS_LABELS)
quantity_mix = quantity_counts.div(quantity_counts.sum(axis=1), axis=0)
display(quantity_counts)
display(quantity_mix.style.format('{:.1%}'))

,bias-corrected Cramér V
quantity,0.309
extraction_type_class,0.241
waterpoint_type_group,0.227
region,0.200
payment_type,0.182
water_quality,0.138
source_class,0.070
management_group,0.049


status_group,functional,functional needs repair,non functional
quantity,,,
dry,157,37,6052
enough,21648,2400,9138
insufficient,7916,1450,5763
seasonal,2325,416,1309
unknown,213,14,562


status_group,functional,functional needs repair,non functional
quantity,,,
dry,2.5%,0.6%,96.9%
enough,65.2%,7.2%,27.5%
insufficient,52.3%,9.6%,38.1%
seasonal,57.4%,10.3%,32.3%
unknown,27.0%,1.8%,71.2%


In [5]:
discordance_summary = pd.DataFrame(
    {
        'rows': [
            int((
                labelled['quantity'].eq('dry')
                & labelled['status_group'].ne('non functional')
            ).sum()),
            int((
                labelled['quantity'].eq('enough')
                & labelled['status_group'].eq('non functional')
            ).sum()),
        ],
        'why this is not automatically a label error': [
            'Quantity and status are subjective/contextual; dry may be seasonal or a field-code mismatch.',
            'Water availability does not prove the extraction hardware was delivering service.',
        ],
    },
    index=[
        'dry quantity but an operational-labelled class',
        'enough quantity but non-functional label',
    ],
)
display(discordance_summary)

predictor_columns = [column for column in raw_training if column != 'id']
duplicate_mask = raw_training.duplicated(
    subset=predictor_columns,
    keep=False,
)
duplicate_rows = labelled.loc[duplicate_mask]
duplicate_groups = duplicate_rows.groupby(
    predictor_columns,
    dropna=False,
)
labels_per_duplicate_group = duplicate_groups['status_group'].nunique()
duplicate_summary = pd.Series(
    {
        'rows in exact-predictor duplicate groups': int(duplicate_mask.sum()),
        'exact-predictor duplicate groups': int(duplicate_groups.ngroups),
        'groups containing conflicting labels': int(
            labels_per_duplicate_group.gt(1).sum()
        ),
    },
    name='count',
).to_frame()
display(duplicate_summary)

,rows,why this is not automatically a label error
dry quantity but an operational-labelled class,194,Quantity and status are subjective/contextual;...
enough quantity but non-functional label,9138,Water availability does not prove the extracti...


,count
rows in exact-predictor duplicate groups,72
exact-predictor duplicate groups,35
groups containing conflicting labels,1


The exact-predictor comparison excludes the competition ID and uses all 39 supplied predictors. Thirty-five indistinguishable groups are a small fraction of the data, but one group contains more than one label. A deterministic model using only supplied predictors cannot classify every member of that group correctly. The rows might be duplicated records, distinct co-located water points, stale observations, or missing distinguishing information; the competition extract cannot adjudicate. This is direct evidence of at least some irreducible ambiguity, not a licence to rewrite labels.

## 6. Which target structure fits?

### Flat nominal multiclass

This is the safest competition contract. DrivenData accepts one of three names and scores classification accuracy, so every wrong class costs the same. Flat treatment does not claim the classes are conceptually unrelated; it avoids imposing unverified geometry.

### Simple ordinal encoding

The order functional < functional needs repair < non functional is intuitively plausible as increasing service disruption. It is only a hypothesis. The source programme mixed current flow, hardware condition and breakdown duration; a dry or abandoned point is not necessarily one scalar step beyond a cheap repair. Numeric codes 0/1/2 would also invent equal spacing unless an ordinal method uses thresholds without assuming distances. The two cumulative thresholds correspond to intervention-needed versus functional and non-functional versus the two other classes, but learnable thresholds do not establish a true latent severity scale.

### Tree A: wording/operational semantics

1. Is the water point operational? If no, assign non functional.
2. If operational, are repairs needed? If yes, assign functional needs repair; otherwise functional.

Tree A follows DrivenData's wording. Its source validity is imperfect because the historical programme could assign recently non-functional points to repair-needed or even functional.

### Tree B: maintenance-decision semantics

1. Is an intervention needed? If no, assign functional.
2. If intervention is needed, is the point non-operational? If yes, assign non functional; otherwise functional needs repair.

Tree B matches a maintenance triage decision and gives the minority repair class its own branch. It assumes both non-functional and repair-needed points call for intervention, which the label does not guarantee: it contains no repair cost, feasibility, cause, urgency or beneficiary count.

### Multiple latent axes

This is the best conceptual description of the source process: operational service now; hardware defect; repair requirement; breakdown duration; quantity/seasonality; quality; and administrative reporting incentives. status_group compresses those axes into one competition label. A future operational system should collect the axes separately, then derive task-specific labels under a versioned rule.

## 7–9. Frozen-fold separability and structural probes

The following diagnostic reuses the frozen development partition, five predefined folds, existing 36-predictor policy, fold-fitted feature engineering and preprocessing, and the repository's constrained depth-12 decision tree. The untouched local test is not inspected.

This deliberately modest model answers a structural question: which decisions are comparatively easy to reproduce from the supplied fields? It is not a fresh model-family search and its scores must not be compared as if they were final-model estimates. Branch models are trained only on labels belonging to their branch.

In [6]:
binary_tasks = {
    'A root: operational vs non-operational': {
        'positive': lambda target: target.ne('non functional'),
        'include': None,
        'positive_label': 'operational-labelled',
    },
    'A branch: repair vs no repair': {
        'positive': lambda target: target.eq('functional needs repair'),
        'include': lambda target: target.ne('non functional'),
        'positive_label': 'functional needs repair',
    },
    'B root: intervention vs no intervention': {
        'positive': lambda target: target.ne('functional'),
        'include': None,
        'positive_label': 'repair or non-functional',
    },
    'B branch: non-operational vs repair': {
        'positive': lambda target: target.eq('non functional'),
        'include': lambda target: target.ne('functional'),
        'positive_label': 'non functional',
    },
    'Pair: non-functional vs functional': {
        'positive': lambda target: target.eq('non functional'),
        'include': lambda target: target.ne('functional needs repair'),
        'positive_label': 'non functional',
    },
}

binary_fold_rows = []
task_probabilities = {
    task_name: np.full(len(partitioned.y_development), np.nan)
    for task_name in binary_tasks
}

for task_name, task in binary_tasks.items():
    for fold_number, (training_positions, validation_positions) in enumerate(
        fold_splits,
        start=1,
    ):
        training_target = partitioned.y_development.iloc[training_positions]
        validation_target = partitioned.y_development.iloc[validation_positions]
        if task['include'] is None:
            training_mask = np.ones(len(training_positions), dtype=bool)
            validation_mask = np.ones(len(validation_positions), dtype=bool)
        else:
            training_mask = task['include'](training_target).to_numpy()
            validation_mask = task['include'](validation_target).to_numpy()

        task_training_positions = training_positions[training_mask]
        task_validation_positions = validation_positions[validation_mask]
        y_training = task['positive'](
            partitioned.y_development.iloc[task_training_positions]
        ).astype(int)
        y_validation = task['positive'](
            partitioned.y_development.iloc[task_validation_positions]
        ).astype(int)

        pipeline = make_initial_decision_tree_pipeline()
        pipeline.fit(
            partitioned.X_development.iloc[task_training_positions],
            y_training,
        )
        classifier = pipeline.named_steps['classifier']
        positive_column = list(classifier.classes_).index(1)
        all_validation_probabilities = pipeline.predict_proba(
            partitioned.X_development.iloc[validation_positions]
        )[:, positive_column]
        task_probabilities[task_name][validation_positions] = (
            all_validation_probabilities
        )

        probabilities = all_validation_probabilities[validation_mask]
        predictions = probabilities >= 0.5
        negative_recall, positive_recall = recall_score(
            y_validation,
            predictions,
            labels=[0, 1],
            average=None,
            zero_division=0,
        )
        binary_fold_rows.append(
            {
                'task': task_name,
                'positive label': task['positive_label'],
                'fold': fold_number,
                'evaluated rows': len(task_validation_positions),
                'positive rows': int(y_validation.sum()),
                'accuracy': accuracy_score(y_validation, predictions),
                'balanced accuracy': balanced_accuracy_score(
                    y_validation,
                    predictions,
                ),
                'ROC AUC': roc_auc_score(y_validation, probabilities),
                'negative recall': negative_recall,
                'positive recall': positive_recall,
            }
        )

binary_fold_metrics = pd.DataFrame(binary_fold_rows)
binary_probe_summary = binary_fold_metrics.groupby(
    ['task', 'positive label'],
).agg(
    evaluated_rows=('evaluated rows', 'sum'),
    positive_rows=('positive rows', 'sum'),
    accuracy=('accuracy', 'mean'),
    balanced_accuracy=('balanced accuracy', 'mean'),
    roc_auc=('ROC AUC', 'mean'),
    negative_recall=('negative recall', 'mean'),
    positive_recall=('positive recall', 'mean'),
)
binary_probe_summary['positive_share'] = (
    binary_probe_summary['positive_rows']
    / binary_probe_summary['evaluated_rows']
)
binary_probe_summary = binary_probe_summary[[
    'evaluated_rows',
    'positive_share',
    'accuracy',
    'balanced_accuracy',
    'roc_auc',
    'negative_recall',
    'positive_recall',
]]
display(
    binary_probe_summary.style.format(
        {
            'positive_share': '{:.1%}',
            'accuracy': '{:.3f}',
            'balanced_accuracy': '{:.3f}',
            'roc_auc': '{:.3f}',
            'negative_recall': '{:.3f}',
            'positive_recall': '{:.3f}',
        }
    )
)

,,evaluated_rows,positive_share,accuracy,balanced_accuracy,roc_auc,negative_recall,positive_recall
task,positive label,,,,,,,
A branch: repair vs no repair,functional needs repair,29261,11.8%,0.894,0.622,0.792,0.979,0.265
A root: operational vs non-operational,operational-labelled,47520,61.6%,0.806,0.769,0.858,0.610,0.928
B branch: non-operational vs repair,non functional,21713,84.1%,0.885,0.716,0.878,0.469,0.963
B root: intervention vs no intervention,repair or non-functional,47520,45.7%,0.765,0.753,0.835,0.892,0.613
Pair: non-functional vs functional,non functional,44066,41.4%,0.802,0.780,0.866,0.906,0.654


In [7]:
flat_predictions = np.empty(len(partitioned.y_development), dtype=object)
for training_positions, validation_positions in fold_splits:
    flat_pipeline = make_initial_decision_tree_pipeline()
    flat_pipeline.fit(
        partitioned.X_development.iloc[training_positions],
        partitioned.y_development.iloc[training_positions],
    )
    flat_predictions[validation_positions] = flat_pipeline.predict(
        partitioned.X_development.iloc[validation_positions]
    )

a_root = task_probabilities['A root: operational vs non-operational']
a_branch = task_probabilities['A branch: repair vs no repair']
tree_a_predictions = np.where(
    a_root < 0.5,
    'non functional',
    np.where(
        a_branch >= 0.5,
        'functional needs repair',
        'functional',
    ),
)

b_root = task_probabilities['B root: intervention vs no intervention']
b_branch = task_probabilities['B branch: non-operational vs repair']
tree_b_predictions = np.where(
    b_root < 0.5,
    'functional',
    np.where(
        b_branch >= 0.5,
        'non functional',
        'functional needs repair',
    ),
)

structural_prediction_sets = {
    'flat multiclass': flat_predictions,
    'Tree A: operational first': tree_a_predictions,
    'Tree B: intervention first': tree_b_predictions,
}
structural_rows = []
for structure_name, predictions in structural_prediction_sets.items():
    class_recalls = recall_score(
        partitioned.y_development,
        predictions,
        labels=CLASS_LABELS,
        average=None,
        zero_division=0,
    )
    structural_rows.append(
        {
            'structure': structure_name,
            'accuracy': accuracy_score(
                partitioned.y_development,
                predictions,
            ),
            'macro recall': class_recalls.mean(),
            **{
                f'recall: {label}': value
                for label, value in zip(
                    CLASS_LABELS,
                    class_recalls,
                    strict=True,
                )
            },
        }
    )
structural_comparison = pd.DataFrame(structural_rows).set_index('structure')
display(structural_comparison.style.format('{:.3f}'))

,accuracy,macro recall,recall: functional,recall: functional needs repair,recall: non functional
structure,,,,,
flat multiclass,0.750,0.566,0.908,0.152,0.640
Tree A: operational first,0.748,0.588,0.914,0.240,0.610
Tree B: intervention first,0.748,0.574,0.892,0.178,0.651


### Proper ordinal cumulative-threshold probe

The tempting ordinal story is now tested directly rather than merely discussed. Encode only the **order**, not equal spacing: $Y \geq 1$ means repair-or-worse and $Y \geq 2$ means non-functional. Two binary models estimate those cumulative probabilities. Independent estimates can cross, so crossing pairs are projected to their equal-weight mean before reconstructing three non-negative class probabilities.

Three decisions are recorded: maximum reconstructed class probability; two pre-specified 0.5 cut-offs; and two cut-offs selected by nominal accuracy on a calibration fold nested inside each outer training partition. The last scheme is the disciplined version of threshold 'fiddling': every reported outer-fold row was absent from both model fitting and cut-off selection. The cut-off grid is 0.20–0.80 in 0.05 steps, with balanced accuracy and closeness to 0.5 used only as tie-breakers. The local test remains untouched.

In [8]:
repair_or_worse = task_probabilities[
    'B root: intervention vs no intervention'
]
non_functional = 1.0 - task_probabilities[
    'A root: operational vs non-operational'
]
crossing_mask = non_functional > repair_or_worse
ordinal_probabilities = cumulative_class_probabilities(
    repair_or_worse,
    non_functional,
)
ordinal_probability_predictions = np.asarray(CLASS_LABELS)[
    ordinal_probabilities.argmax(axis=1)
]
ordinal_default_predictions = cumulative_threshold_predictions(
    repair_or_worse,
    non_functional,
)

def fit_cumulative_roots(training_positions, prediction_positions):
    training_target = partitioned.y_development.iloc[training_positions]
    prediction_values = []
    for positive_target, invert in (
        (training_target.ne('functional').astype(int), False),
        (training_target.ne('non functional').astype(int), True),
    ):
        pipeline = make_initial_decision_tree_pipeline()
        pipeline.fit(
            partitioned.X_development.iloc[training_positions],
            positive_target,
        )
        positive_column = list(
            pipeline.named_steps['classifier'].classes_
        ).index(1)
        values = pipeline.predict_proba(
            partitioned.X_development.iloc[prediction_positions]
        )[:, positive_column]
        prediction_values.append(1.0 - values if invert else values)
    return tuple(prediction_values)

ordinal_calibrated_predictions = np.empty(
    len(partitioned.y_development),
    dtype=object,
)
cutoff_rows = []
for outer_fold, (training_positions, validation_positions) in enumerate(
    fold_splits,
    start=1,
):
    calibration_fold = outer_fold % len(fold_splits) + 1
    training_fold_numbers = partitioned.validation_folds.iloc[
        training_positions
    ].to_numpy()
    calibration_positions = training_positions[
        training_fold_numbers == calibration_fold
    ]
    inner_training_positions = training_positions[
        training_fold_numbers != calibration_fold
    ]
    calibration_repair, calibration_non_functional = (
        fit_cumulative_roots(
            inner_training_positions,
            calibration_positions,
        )
    )
    selected = tune_cumulative_cutoffs(
        partitioned.y_development.iloc[calibration_positions],
        calibration_repair,
        calibration_non_functional,
    )
    ordinal_calibrated_predictions[validation_positions] = (
        cumulative_threshold_predictions(
            repair_or_worse[validation_positions],
            non_functional[validation_positions],
            repair_cutoff=selected['repair_cutoff'],
            non_functional_cutoff=selected['non_functional_cutoff'],
        )
    )
    cutoff_rows.append(
        {
            'outer fold': outer_fold,
            'calibration fold': calibration_fold,
            'inner training rows': len(inner_training_positions),
            'calibration rows': len(calibration_positions),
            **selected,
        }
    )

ordinal_prediction_sets = {
    **structural_prediction_sets,
    'Ordinal: reconstructed probability argmax': (
        ordinal_probability_predictions
    ),
    'Ordinal: two fixed 0.5 cut-offs': ordinal_default_predictions,
    'Ordinal: nested calibrated cut-offs': (
        ordinal_calibrated_predictions
    ),
}
ordinal_rows = []
for structure_name, predictions in ordinal_prediction_sets.items():
    class_recalls = recall_score(
        partitioned.y_development,
        predictions,
        labels=CLASS_LABELS,
        average=None,
        zero_division=0,
    )
    ordinal_rows.append(
        {
            'structure': structure_name,
            'accuracy': accuracy_score(
                partitioned.y_development, predictions
            ),
            'macro recall': class_recalls.mean(),
            **{
                f'recall: {label}': value
                for label, value in zip(
                    CLASS_LABELS, class_recalls, strict=True
                )
            },
            **ordinal_error_metrics(
                partitioned.y_development, predictions
            ),
        }
    )
ordinal_comparison = pd.DataFrame(ordinal_rows).set_index('structure')
cutoff_selection = pd.DataFrame(cutoff_rows).set_index('outer fold')
ordinal_diagnostics = pd.Series(
    {
        'cumulative probability crossing rows': int(crossing_mask.sum()),
        'cumulative probability crossing share': crossing_mask.mean(),
    },
    name='value',
)
display(ordinal_comparison.style.format('{:.3f}'))
display(cutoff_selection.style.format('{:.3f}'))
display(ordinal_diagnostics.to_frame())

,accuracy,macro recall,recall: functional,recall: functional needs repair,recall: non functional,mean_absolute_ordinal_error,two_step_error_rate
structure,,,,,,,
flat multiclass,0.750,0.566,0.908,0.152,0.640,0.430,0.179
Tree A: operational first,0.748,0.588,0.914,0.240,0.610,0.431,0.179
Tree B: intervention first,0.748,0.574,0.892,0.178,0.651,0.432,0.180
Ordinal: reconstructed probability argmax,0.742,0.573,0.908,0.198,0.611,0.420,0.163
Ordinal: two fixed 0.5 cut-offs,0.724,0.578,0.882,0.265,0.588,0.421,0.146
Ordinal: nested calibrated cut-offs,0.745,0.550,0.925,0.115,0.610,0.426,0.172


,calibration fold,inner training rows,calibration rows,repair_cutoff,non_functional_cutoff,calibration_accuracy,calibration_balanced_accuracy
outer fold,,,,,,,
1,2.000,28512.000,9504.000,0.800,0.450,0.751,0.549
2,3.000,28512.000,9504.000,0.750,0.450,0.742,0.566
3,4.000,28512.000,9504.000,0.800,0.500,0.740,0.549
4,5.000,28512.000,9504.000,0.750,0.450,0.744,0.555
5,1.000,28512.000,9504.000,0.800,0.400,0.747,0.556


,value
cumulative probability crossing rows,13703.000000
cumulative probability crossing share,0.288363


### Interpretation of the recorded probe

The functional-versus-non-functional pair is the cleanest direct separation. The repair/no-repair decision within operational-labelled rows is much harder, especially for the minority positive class. This is consistent with a fuzzy repair construct, under-representation, or missing maintenance-condition variables; it cannot distinguish those explanations.

With the same constrained learner, flat multiclass gives the best accuracy by a small margin, which is exactly the metric the competition rewards. Tree A trades a small amount of accuracy for materially better repair recall and macro recall. Tree B makes a different recall trade-off. These are modelling consequences, not proof that either tree generated the labels. A stronger classifier could change the numerical ordering.

The proper ordinal reduction does not produce an accuracy win with this learner. Reconstructed-probability argmax scores 0.742 accuracy versus 0.750 flat, while lowering mean absolute ordinal error from 0.430 to 0.420 and two-step errors from 17.9% to 16.3%. Fixed 0.5 cut-offs improve repair recall to 26.5% but reduce accuracy to 0.724. Development-only nested cut-off calibration chooses conservative repair cut-offs of 0.75–0.80 and reaches 0.745 accuracy, but repair recall falls to 11.5%.

Most importantly, 28.8% of the independently estimated cumulative probability pairs cross before coherence projection. That is not proof against every ordinal model—the constrained tree and independent-root construction are limited—but it is substantial evidence against the simple 'strongly ordinal, set two thresholds, tune, win' account. Threshold adjustment changes error costs; it does not manufacture a clean one-dimensional target.

## Prediction-time semantics and label/proxy risks

1. **Snapshot staleness:** status can change after repair, breakdown, seasonal source change or abandonment. A historic row is not current truth.
2. **Concept drift:** later Tanzanian reporting programmes used revised conditional definitions. A model can reproduce the old rule while disagreeing with a current inspector.
3. **Observer and informant discretion:** source research reports local negotiation and incentives around status, plus ambiguity in definitions.
4. **Same-visit proxies:** quantity and quality may have been judged during the same visit. They are fair competition predictors but may be unavailable before an inspection and may partly encode the reporter's reasoning.
5. **Geographic/institutional proxies:** region, management, installer and technology can encode different failure environments and different data-collection practices. Random folds estimate interpolation within that historic national snapshot, not transfer to a new country or era.
6. **Missing causal/action fields:** the competition lacks a verified fault diagnosis, repair cost, parts availability, repair success, time since failure and beneficiary harm. status_group is therefore unsuitable as a stand-alone rehabilitation priority.
7. **Equal error cost:** competition accuracy treats every class confusion equally. Operations normally do not; missing a non-functional point or wasting a site visit has a context-dependent cost.

For a real deployment, define an as-of time, decide whether same-visit measurements are available, publish a versioned status rubric, capture elemental fields separately, collect repeated labels and evaluate temporal/geographic holdouts plus action-specific costs.

## 10. Conclusion and iteration decision

- **Competition modelling:** keep status_group as the authoritative flat nominal target and optimise/compare against the published multiclass metric. Preserve per-class recall and macro metrics so the minority repair class is not hidden.
- **Structural experiments:** retain both trees as explicit alternatives. Tree A represents DrivenData's operational wording; Tree B represents a maintenance-intervention decision. Neither should silently replace the competition task.
- **Ordinal experiments:** the proper cumulative-threshold probe reduces ordinal-distance errors but does not beat flat accuracy; nested cut-off tuning also fails to win. Do not feed 0/1/2 into ordinary regression or treat a single latent severity scale as established.
- **Ontology:** the most faithful account is multiple latent axes collapsed by a historical administrative rule.
- **Robustness:** do not relabel suspicious rows. Use stratification, fold-local imbalance treatment, class-aware metrics and error analysis. Treat the exact duplicate conflict as evidence of an imperfect ceiling.
- **Deployment:** not applicable in this notebook. The next real-world iteration would require new, timestamped, repeatedly observed labels under a current rubric.

In [9]:
assert len(raw_training) == 59_400
assert len(raw_competition) == 14_850
assert len(all_competition_ids) == 74_250
assert all_competition_ids.nunique() == 74_250
assert set(all_competition_ids) == set(range(74_250))
assert target_counts.to_dict() == {
    'functional': 32_259,
    'functional needs repair': 4_317,
    'non functional': 22_824,
}
assert recorded_dates.dt.year.isin([2011, 2012, 2013]).sum() == 59_369
assert duplicate_groups.ngroups == 35
assert labels_per_duplicate_group.gt(1).sum() == 1
assert binary_fold_metrics.groupby('task')['fold'].nunique().eq(5).all()
assert np.isfinite(binary_probe_summary.to_numpy()).all()
assert np.isfinite(structural_comparison.to_numpy()).all()
assert np.isfinite(ordinal_comparison.to_numpy()).all()
assert ordinal_calibrated_predictions.tolist().count(None) == 0
assert set(ordinal_calibrated_predictions) == set(CLASS_LABELS)
assert len(cutoff_selection) == 5
print('All provenance, partition, diagnostic and recorded-output assertions passed.')

All provenance, partition, diagnostic and recorded-output assertions passed.
